# LLM Reasoning Optimization — Experiments

Toto je **jediný Colab notebook** pro náš projekt. Při novém runtime spusť buňky shora dolů.

**Aktuálně hotovo:** setup prostředí + příprava dat pro **SFT (GSM8K original reasoning)** a **RL (BottleCapAI original)**. SFT trainer už je připravený; evaluaci a GRPO přidáme později jako samostatné Python skripty.

> Notebook je rozhraní; zdrojový kód je ve složce `src/`. GitHub ukládá kód, Google Drive ukládá datasety/modely/výsledky a `/content` slouží jako dočasný pracovní disk.


## 1. Nastavení projektu

Zde nastavíme adresář repozitáře a dat. Pokud už data na Drive máme, `REBUILD_DATA = False` je **nepřipraví znovu**.


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/ondrysL/llm-reasoning-optimization.git"
PROJECT_DIR = Path("/content/llm-reasoning-optimization")
DATA_DIR = Path("/content/drive/MyDrive/llm-reasoning/datasets")
REBUILD_DATA = False  # True: vynutit znovuvytvoření datasetů


## 2. Připojení Google Drive

Drive zůstane úložištěm datasetů i po restartu Colab runtime.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DATA_DIR.mkdir(parents=True, exist_ok=True)
print("Persistent dataset directory:", DATA_DIR)


## 3. Projekt v Colabu: git clone / git pull + instalace

Při prvním spuštění projekt naklonujeme. Při dalších spuštěních ve stejném runtime použijeme `git pull --ff-only`. Python skripty a konfigurace tak vždy bereme z GitHubu.


In [ ]:
import os
import subprocess
import sys


def run(*command, cwd=None):
    command = [str(part) for part in command]
    print("+", " ".join(command), flush=True)
    subprocess.run(command, cwd=cwd, check=True)


if (PROJECT_DIR / ".git").is_dir():
    run("git", "-C", PROJECT_DIR, "pull", "--ff-only")
elif PROJECT_DIR.exists():
    raise RuntimeError(f"{PROJECT_DIR} already exists but is not a Git checkout")
else:
    run("git", "clone", REPO_URL, PROJECT_DIR)

os.chdir(PROJECT_DIR)
run(sys.executable, "-m", "pip", "install", "-e", ".", cwd=PROJECT_DIR)
print("Project ready:", PROJECT_DIR)


## 4. BottleCapAI repozitář

Načteme **konkrétní připnutou verzi** BottleCapAI do `external/ThinkingCap-Test` (neměníme upstream kód). Jde o malý checkout zdrojových skriptů, ne o opakované stahování datasetů.


In [ ]:
run("bash", "scripts/bootstrap_bottlecap.sh", cwd=PROJECT_DIR)


## 5. Příprava datasetů (jednou, potom reuse)

Náš `reasoning_opt.data.prepare` spustí původní BottleCapAI přípravu čtyř datasetů a k jeho GSM8K otázkám připojí **originální lidská řešení** pro SFT. Žádný další LLM ani concise rewriting nepoužíváme.

Vytvoří na Drive:

- `bottlecap/train.jsonl`, `bottlecap/val.jsonl` (otázky + odpovědi, zejména pro GRPO),
- `gsm8k-original/train.jsonl`, `gsm8k-original/val.jsonl` (otázky + referenční postupy pro SFT),
- `manifest.json` (počty, verze a kontrolní součty).

První příprava může trvat několik minut v závislosti na stahování z Hugging Face. Při dalších bězích se přeskočí.


In [ ]:
manifest_path = DATA_DIR / "manifest.json"
source_dir = PROJECT_DIR / "external" / "ThinkingCap-Test" / "data"

if manifest_path.is_file() and not REBUILD_DATA:
    print("Datasets already prepared; skipping download and processing.")
    print("Manifest:", manifest_path)
else:
    options = ["--reuse-source"] if (
        (source_dir / "train.jsonl").is_file() and (source_dir / "val.jsonl").is_file()
    ) else []
    run(
        sys.executable, "-m", "reasoning_opt.data.prepare",
        "--upstream", PROJECT_DIR / "external" / "ThinkingCap-Test",
        "--output-dir", DATA_DIR,
        *options,
        cwd=PROJECT_DIR,
    )


## 6. Rychlá kontrola připravených dat

Zkontrolujeme, že soubory existují, počty dávají smysl a GSM8K SFT train a validation neobsahují stejné otázky.


In [ ]:
import json

manifest = json.loads((DATA_DIR / "manifest.json").read_text(encoding="utf-8"))
required_files = [
    "bottlecap/train.jsonl", "bottlecap/val.jsonl",
    "gsm8k-original/train.jsonl", "gsm8k-original/val.jsonl",
]
for name in required_files:
    path = DATA_DIR / name
    assert path.is_file() and path.stat().st_size > 0, f"Missing or empty dataset: {path}"

print("BottleCap commit:", manifest["upstream_commit"])
print("Dataset counts:")
for key, count in manifest["counts"].items():
    print(f"  {key}: {count}")


def read_ids(filename):
    with (DATA_DIR / filename).open(encoding="utf-8") as f:
        return {json.loads(line)["id"] for line in f if line.strip()}


train_ids = read_ids("gsm8k-original/train.jsonl")
val_ids = read_ids("gsm8k-original/val.jsonl")
assert not (train_ids & val_ids), "Train/validation leakage in GSM8K SFT"

with (DATA_DIR / "gsm8k-original/train.jsonl").open(encoding="utf-8") as f:
    example = json.loads(next(f))
assert example["reasoning"].strip() and example["answer"].strip()
print("\nFirst GSM8K SFT example:")
print("Question:", example["prompt"][0]["content"][:350])
print("Reasoning:", example["reasoning"][:400])
print("Answer:", example["answer"])
print("\nData preparation and checks: OK")


## 7. SFT + LoRA — konfigurace a trénování

Tady budeme spouštět konfigurovatelný SFT trainer z `src/reasoning_opt/training/sft.py`. Před první evaluací je důležité ověřit, že target opravdu začíná `<think>`.

1. Nastav experiment v `configs/sft.json` lokálně, pushni Git a v Colabu obnov projekt pomocí původní setup buňky s `git pull`.
2. Instaluj trénovací závislosti; potom spusť preflight. Následující buňka spustí GPU trénování.
3. Uložení probíhá do `MyDrive/llm-reasoning/experiments`, *ne* do dočasného `/content`.

**Pozn.:** první trénink je nastaven na 1 000 / 100 příkladů; po úspěšném otestování konfiguraci zvětšíme.


In [ ]:
# Jednou za Colab runtime (T4 GPU) nainstalujeme HF trainer, PEFT a TensorBoard.
run(sys.executable, "-m", "pip", "install", "-e", ".[training]", cwd=PROJECT_DIR)


In [ ]:
# Než spustíme GPU: kontrola splitů, délky sekvencí a Qwen3 <think> formátu.
SFT_CONFIG = PROJECT_DIR / "configs" / "sft.json"
print(SFT_CONFIG.read_text(encoding="utf-8"))
run(sys.executable, "-u", "-m", "reasoning_opt.training.sft",
    "--config", SFT_CONFIG, "--preflight-only", cwd=PROJECT_DIR)


In [ ]:
# This helper streams subprocess logs line-by-line, including tracebacks.
# The scripts themselves remain inside our versioned Python project.
import subprocess
import sys

def stream_run(*args, cwd=PROJECT_DIR):
    cmd = [str(part) for part in args]
    print("+", " ".join(cmd), flush=True)
    proc = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    rc = proc.wait()
    if rc:
        raise RuntimeError(f"Command failed with exit code {rc}: {cmd}")


In [ ]:
# Run one SFT configuration. Never start this while another GPU training is active.
stream_run(sys.executable, "-u", "-m", "reasoning_opt.training.sft",
           "--config", SFT_CONFIG)


### Trénovací grafy (volitelné)

Až se objeví první logy, spusť následující buňku pro TensorBoard grafy trénovacího a validačního lossu.


In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/llm-reasoning/experiments


## 8. Training multiple LoRA configurations (optional)

The original SFT cell runs **one** JSON config. If we want *several independent* models, list them in `configs/sft_batch.json`. They will run **sequentially** (one GPU). Re-running the batch creates new timestamped training runs for **all configs listed**. Do not run this while another training is active.


In [ ]:
# Check batch configurations without training.
stream_run(sys.executable, "-u", "-m", "reasoning_opt.training.batch",
           "--config", PROJECT_DIR / "configs/sft_batch.json", "--preflight-only")


In [ ]:
# Uncomment to train all configurations in sft_batch.json (sequentially).
# stream_run(sys.executable, "-u", "-m", "reasoning_opt.training.batch",
#            "--config", PROJECT_DIR / "configs/sft_batch.json")


## 9. Development evaluation: GSM8K validation (for selecting hyperparameters)

The code runs the original Qwen baseline and **all completed LoRA experiments on Google Drive**. It uses 100 held-out GSM8K questions (configurable up to 200), with greedy decoding. It reports accuracy, number of generated tokens, thinking format errors and reasoning length. This is our **development set, not BottleCap's official benchmark**.

Set `experiment_names: []` in `configs/evaluation.json` for every completed run, or explicitly list exact timestamped experiment folder names. **Warning:** this can take several minutes per model on T4.


In [ ]:
EVAL_CONFIG = PROJECT_DIR / "configs" / "evaluation.json"
print(EVAL_CONFIG.read_text(encoding="utf-8"))
# All LoRA runs are now saved, so they can be evaluated later without retraining.
stream_run(sys.executable, "-u", "-m", "reasoning_opt.evaluation.dev",
           "--config", EVAL_CONFIG)


## 10. BottleCapAI benchmark: smoke, preliminary, official

Use **only after model selection using validation data**. The upstream benchmark is a test set; avoid tuning hyperparameters on its questions. Our wrapper uses the original BottleCap scoring scripts but explicitly chooses `TRITON_ATTN` and float16 in a copied evaluator for Colab T4.

Installation is large. Recommended first run: `mode: "smoke"`, `benchmarks: ["svamp"]`, `samples: 1`. It checks only 5 prompts and **does not yield valid scores**. For a preliminary SVAMP check use `mode: "benchmark"` and `samples: 1`. A full official evaluation uses all four benchmarks and eight generations per question (slow on T4).

**MBPP+ safety:** its code grader executes generated Python. On Colab, missing bubblewrap may cause BottleCap to fall back to less isolated resource limits. Run only in an appropriate disposable environment.


In [ ]:
# Only needed when using BottleCap benchmarking (not the development validator).
# This can download vLLM and additional CUDA dependencies.
stream_run(sys.executable, "-m", "pip", "install", "-r",
           PROJECT_DIR / "external/ThinkingCap-Test/requirements.txt",
           "vllm==0.23.0")


In [ ]:
# Config mode: "smoke" or "benchmark". Reads completed LoRA adapters from Drive.
stream_run(sys.executable, "-u", "-m", "reasoning_opt.evaluation.benchmark",
           "--config", EVAL_CONFIG)


## 11. RL / GRPO — later

After we verify SFT + development evaluation and final accuracy/length tradeoffs, we will implement a separate GRPO trainer with correct-answer and length rewards. No teacher LLM is necessary.
